In [1]:
import os
import csv
import pandas as pd
import numpy as np
import math

In [12]:
def human_ent(df):
    def ent_calc(arr):
        arr = np.asarray(arr, dtype=float)

        total = arr.sum()
        if total == 0:
            return 0.0  # handle all-zero case

        p = arr / total
        p = p[p > 0]  # avoid log(0)

        H = -np.sum(p * np.log(p))
        H_max = np.log(len(arr))

        return H / H_max if H_max > 0 else 0.0

    tmp_df = df.copy()
    tmp_df['human-ent'] = tmp_df['Answer_Ratios'].apply(ent_calc)
    return tmp_df

def update_df(init_df : pd.DataFrame) -> pd.DataFrame:
    results = []

    fr_table = pd.read_csv('./table_A1.csv')
    mc_table = pd.read_csv('./table_A2.csv')

    for (_, fr_row), (_, mc_row) in zip(fr_table.iterrows(), mc_table.iterrows()):
        assert fr_row['General Knowledge Question'] == mc_row['Question']

        fr_cr_p = fr_row['CR (proportion of responses) – Correct']
        fr_ce_p = fr_row['CR (proportion of responses) – CE']
        fr_dr_p = fr_row['CR (proportion of responses) – DR']
        fr_dk_p = fr_row['CR (proportion of responses) – DK']

        fr_cr_rt = fr_row['CR (response times) – Correct']
        fr_ce_rt = fr_row['CR (response times) – CE']
        fr_dr_rt = fr_row['CR (response times) – DR']
        fr_dk_rt = fr_row['CR (response times) – DK']

        mc_cr_p = mc_row['Correct']
        mc_ce_p = 1 - mc_cr_p

        mc_cr_rt = mc_row['MC  (response times) – Correct']
        mc_ce_rt = mc_row['MC  (response times) – Errors']

        # Renormalize out OE
        fr_non_oe = fr_cr_p + fr_dr_p + fr_dk_p + fr_ce_p
        fr_cr_p /= fr_non_oe
        fr_ce_p /= fr_non_oe
        fr_dr_p /= fr_non_oe
        fr_dk_p /= fr_non_oe

        # Combine FR_CR + FR_CE
        fr_ans_p = fr_cr_p + fr_ce_p
        if fr_ans_p == float(0.0):
            fr_ans_rt = float('nan')
        else:
            fr_ans_cr_p = fr_cr_p / fr_ans_p
            fr_ans_ce_p = fr_ce_p / fr_ans_p
            if fr_ans_cr_p == float(0.0):
                fr_ans_rt = fr_ce_rt
            elif fr_ans_ce_p == float(0.0):
                fr_ans_rt = fr_cr_rt
            else:
                fr_ans_rt = (fr_ans_cr_p * fr_cr_rt) + (fr_ans_ce_p * fr_ce_rt)

        # Combine FR_DK + FR_DR
        fr_dna_p = fr_dk_p + fr_dr_p
        
        if fr_dna_p == float(0.0):
            fr_dna_rt = float('nan')
        else:
            fr_dna_dk_p = fr_dk_p / fr_dna_p
            fr_dna_dr_p = fr_dr_p / fr_dna_p
            if fr_dna_dk_p == float(0.0):
                fr_dna_rt = fr_dr_rt
            elif fr_dna_dr_p == float(0.0):
                fr_dna_rt = fr_dk_rt
            else:
                fr_dna_rt = (fr_dna_dk_p * fr_dk_rt) + (fr_dna_dr_p * fr_dr_rt)

        # Combine MC_CR + MC_CE
        if mc_cr_p == float(0.0):
            mc_comb_rt = mc_ce_rt
        elif mc_ce_p == float(0.0):
            mc_comb_rt = mc_cr_rt
        else:
            mc_comb_rt = (mc_cr_p * mc_cr_rt) + (mc_ce_p * mc_ce_rt)

        results.append({
            'FR_Human_Prop_Correct'  : fr_cr_p,
            'FR_Human_Prop_Wrong'    : fr_ce_p,
            'FR_Human_Prop_DK'       : fr_dk_p,
            'FR_Human_Prop_DR'       : fr_dr_p,
            'FR_Human_RT_Correct'    : fr_cr_rt,
            'FR_Human_RT_Wrong'      : fr_ce_rt,
            'FR_Human_RT_DK'         : fr_dk_rt,
            'FR_Human_RT_DR'         : fr_dr_rt,
            'FR_Human_Prop_ANS'      : fr_ans_p,
            'FR_Human_Prop_DNA'      : fr_dna_p,
            'FR_Human_RT_ANS'        : fr_ans_rt,
            'FR_Human_RT_DNA'        : fr_dna_rt,
            'MC_Human_Prop_Correct'  : mc_cr_p,
            'MC_Human_Prop_Wrong'    : mc_ce_p,
            'MC_Human_RT_Correct'    : mc_cr_rt,
            'MC_Human_RT_Wrong'      : mc_ce_rt,
            'MC_Human_RT_Combined'   : mc_comb_rt,
        })

    new_df : pd.DataFrame = init_df.assign(**pd.DataFrame(results))

    assert (
        (new_df['FR_Human_RT_Correct'] == new_df['FR_Response_Time'])
        ^ #XOR
        (new_df['FR_Human_RT_Correct'].isna() & new_df['FR_Response_Time'].isna())
    ).all()

    new_df = new_df.drop(['MC_Response_Time','FR_Response_Time','MC_Correct_Human','FR_Correct_Human',], axis=1)
    
    return new_df

def fix_column_mistakes(df):
    cols = [
        ('FR_Human_Prop_Correct', 'FR_Human_RT_Correct', 'FR_Human_RT_Wrong'),
        ('MC_Human_Prop_Correct', 'MC_Human_RT_Correct', 'MC_Human_RT_Wrong'),
    ]
    tmp_df = df.copy()
    for x, y, z in cols:
        mask = (tmp_df[x] == 1.00) & (tmp_df[y].isna())
        num_updated = mask.sum()
        if num_updated > 0:
            tmp_df.loc[mask, [y, z]] = tmp_df.loc[mask, [z, y]].values
            print(f'Fixed misplaced RT in {num_updated} rows')
    return tmp_df

comb_proc = lambda x: update_df(human_ent(fix_column_mistakes(x)))

base_dir = '../../Analysis/Results/coane/'
for fn in os.listdir(base_dir):
    # update_df(pd.read_parquet(base_dir+fn)).to_parquet(base_dir+fn)
    comb_proc(pd.read_parquet(base_dir+fn)).to_parquet(base_dir+fn)
    print(f'Successfully updated {fn}')

KeyError: 'FR_Response_Time'

In [17]:
fix_column_mistakes(pd.read_parquet('coane_data.parquet'))

,Question,Answers,Answer_Ratios,Correct_Answer,Correct_Answer_Text,FR_Human_Prop_Correct,FR_Human_Prop_Wrong,FR_Human_Prop_DK,FR_Human_Prop_DR,FR_Human_RT_Correct,...,FR_Human_Prop_ANS,FR_Human_Prop_DNA,FR_Human_RT_ANS,FR_Human_RT_DNA,MC_Human_Prop_Correct,MC_Human_Proc_Wrong,MC_Human_RT_Correct,MC_Human_RT_Wrong,MC_Human_RT_Combined,human-ent
0,Which band was Paul McCartney a member of?,"[The Monkees, The Rolling Stones, The Yardbird...","[0.02, 0.02, 0.0, 0.96]",D,The Beatles,0.980000,0.020000,0.000000,0.000000,8611.0,...,1.000000,0.000000,8530.420000,NaN,0.96,0.04,4329.0,8795.0,4507.64,0.141146
1,What is the short pleated skirt worn by Scotti...,"[Kelt, Shendyt, Glengarry, Kilt]","[0.204, 0.02, 0.0, 0.78]",D,Kilt,0.960000,0.040000,0.000000,0.000000,7763.0,...,1.000000,0.000000,7921.800000,NaN,0.78,0.22,4757.0,5279.0,4871.84,0.431324
2,"What is the hard, white material sourced from ...","[Enamel, Ivory, Porcelain, Baleen]","[0.0, 1.0, 0.0, 0.0]",B,Ivory,0.940000,0.020000,0.040000,0.000000,6901.0,...,0.960000,0.040000,6952.770833,14494.000000,1.00,0.00,5265.0,NaN,5265.00,-0.000000
3,What word means to trade by exchanging goods f...,"[Haggle, Swap, Export, Barter]","[0.02, 0.061, 0.0, 0.92]",D,Barter,0.939394,0.040404,0.020202,0.000000,8453.0,...,0.979798,0.020202,9454.443299,29489.000000,0.92,0.08,7562.0,9128.0,7687.28,0.235329
4,What is an airplane without an engine called?,"[Biplane, Airship, Glider, Prop]","[0.041, 0.02, 0.92, 0.02]",C,Glider,0.890000,0.020000,0.070000,0.020000,6880.0,...,0.910000,0.090000,7444.065934,7208.777778,0.92,0.08,5656.0,11400.0,6115.52,0.263140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,Who invented the Christmas Cracker?,"[Tom Smith, Charles Plimpton, Frank Hornby, Ol...","[0.04, 0.367, 0.265, 0.327]",A,Tom Smith,0.000000,0.163265,0.836735,0.000000,NaN,...,0.163265,0.836735,23231.000000,5964.000000,0.04,0.96,3034.0,9553.0,9292.24,0.875928
417,What was raced in the first competitive event ...,"[Balloons, Cars, Bicycles, Horses]","[0.02, 0.714, 0.163, 0.102]",A,Balloons,0.000000,0.490000,0.470000,0.040000,NaN,...,0.490000,0.510000,16755.000000,9852.196078,0.02,0.98,201.0,10291.0,10089.20,0.611083
418,What do you call the accumulation of earth and...,"[Moraine, Cirque, Tarn, Esker]","[0.35, 0.102, 0.367, 0.184]",A,Moraine,0.000000,0.297030,0.514851,0.188119,NaN,...,0.297030,0.702970,16696.000000,15154.845070,0.35,0.65,11239.0,15983.0,14322.60,0.922464
419,What is the name of the baseball player with t...,"[DiMaggio, Aaron, Cobb, Mays]","[0.224, 0.327, 0.27, 0.184]",C,Cobb,0.000000,0.680000,0.240000,0.080000,NaN,...,0.680000,0.320000,17631.000000,14910.250000,0.27,0.73,12323.0,11743.0,11899.60,0.983802


In [81]:
os.listdir('../../Analysis/Results/coane/')

['coane_data.parquet',
 'coane_Falcon3_10B.parquet',
 'coane_Falcon3_10B_Ins.parquet',
 'coane_Falcon3_1B.parquet',
 'coane_Falcon3_1B_Ins.parquet',
 'coane_Falcon3_3B.parquet',
 'coane_Falcon3_3B_Ins.parquet',
 'coane_Falcon3_7B.parquet',
 'coane_Falcon3_7B_Ins.parquet',
 'coane_Gemma2_2B.parquet',
 'coane_Gemma2_2B_Ins.parquet',
 'coane_Gemma2_9B.parquet',
 'coane_Gemma2_9B_Ins.parquet',
 'coane_Gemma_2B.parquet',
 'coane_Gemma_2B_Ins.parquet',
 'coane_Gemma_7B.parquet',
 'coane_Gemma_7B_Ins.parquet',
 'coane_Llama_2_13B.parquet',
 'coane_Llama_2_13B_Chat.parquet',
 'coane_Llama_2_7B.parquet',
 'coane_Llama_2_7B_Chat.parquet',
 'coane_Llama_3.1_8B.parquet',
 'coane_Llama_3.1_8B_Ins.parquet',
 'coane_Llama_3.2_1B.parquet',
 'coane_Llama_3.2_1B_Ins.parquet',
 'coane_Llama_3.2_3B.parquet',
 'coane_Llama_3.2_3B_Ins.parquet',
 'coane_Mistral_0.1.parquet',
 'coane_Mistral_0.1_Ins.parquet',
 'coane_Mistral_0.3.parquet',
 'coane_Mistral_0.3_Ins.parquet',
 'coane_Pythia_1.4B.parquet',
 'coane

In [ ]:
pd.read_parquet('../../Analysis/Results/coane/coane_Falcon3_1B.parquet')

,Question,Answers,Answer_Ratios,Correct_Answer,Correct_Answer_Text,FR_Correct_Human,MC_Correct_Human,FR_Response_Time,MC_Response_Time,formatted_question_mcqa,...,bfr_ppl,fr_model_answer,fr_gen_length,fr_hit_cap,fr_bracket_found,fr_bracket_inferred,exp_time,model,model_type,model_size
0,Which band was Paul McCartney a member of?,"[The Monkees, The Rolling Stones, The Yardbird...","[0.02, 0.02, 0.0, 0.96]",D,The Beatles,0.98,0.96,8611.0,4329.0,Question: What is the chemical symbol for gold...,...,1.248102,The Beatles}.\n\nWhat is the chemical symbol f...,100,True,True,False,76.272993,Falcon3_1B,base,1
1,What is the short pleated skirt worn by Scotti...,"[Kelt, Shendyt, Glengarry, Kilt]","[0.204, 0.02, 0.0, 0.78]",D,Kilt,0.96,0.78,7763.0,4757.0,Question: What is the chemical symbol for gold...,...,4.838906,Scotsman}.\n\nWhat is the chemical symbol for ...,100,True,True,False,76.272993,Falcon3_1B,base,1
2,"What is the hard, white material sourced from ...","[Enamel, Ivory, Porcelain, Baleen]","[0.0, 1.0, 0.0, 0.0]",B,Ivory,0.94,1.00,6901.0,5265.0,Question: What is the chemical symbol for gold...,...,3.487515,Tartar}.\n\nWhat is the chemical symbol for go...,100,True,True,False,76.272993,Falcon3_1B,base,1
3,What word means to trade by exchanging goods f...,"[Haggle, Swap, Export, Barter]","[0.02, 0.061, 0.0, 0.92]",D,Barter,0.93,0.92,8453.0,7562.0,Question: What is the chemical symbol for gold...,...,4.125171,trade}.\n\nWhat is the chemical symbol for gol...,100,True,True,False,76.272993,Falcon3_1B,base,1
4,What is an airplane without an engine called?,"[Biplane, Airship, Glider, Prop]","[0.041, 0.02, 0.92, 0.02]",C,Glider,0.89,0.92,6880.0,5656.0,Question: What is the chemical symbol for gold...,...,4.784630,a plane}.\n\nWhat is the chemical symbol for g...,100,True,True,False,76.272993,Falcon3_1B,base,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,Who invented the Christmas Cracker?,"[Tom Smith, Charles Plimpton, Frank Hornby, Ol...","[0.04, 0.367, 0.265, 0.327]",A,Tom Smith,0.00,0.04,NaN,3034.0,Question: What is the chemical symbol for gold...,...,2.898375,18th century}.\n\nWhat is the chemical symbol ...,100,True,True,False,76.272993,Falcon3_1B,base,1
417,What was raced in the first competitive event ...,"[Balloons, Cars, Bicycles, Horses]","[0.02, 0.714, 0.163, 0.102]",A,Balloons,0.00,0.02,NaN,201.0,Question: What is the chemical symbol for gold...,...,1.954489,1909}.\n\nWhat is the chemical symbol for gold...,100,True,True,False,76.272993,Falcon3_1B,base,1
418,What do you call the accumulation of earth and...,"[Moraine, Cirque, Tarn, Esker]","[0.35, 0.102, 0.367, 0.184]",A,Moraine,0.00,0.35,NaN,11239.0,Question: What is the chemical symbol for gold...,...,3.519191,a moraine}.\n\nWhat is the chemical symbol for...,100,True,True,False,76.272993,Falcon3_1B,base,1
419,What is the name of the baseball player with t...,"[DiMaggio, Aaron, Cobb, Mays]","[0.224, 0.327, 0.27, 0.184]",C,Cobb,0.00,0.27,NaN,12323.0,Question: What is the chemical symbol for gold...,...,2.511166,Hank Aaron}.\n\nWhat is the name of the 1990s ...,100,True,True,False,76.272993,Falcon3_1B,base,1


In [5]:
tmp_df

,Question,Answers,Answer_Ratios,Correct_Answer,Correct_Answer_Text,FR_Correct_Human,MC_Correct_Human,FR_Response_Time,MC_Response_Time
0,Which band was Paul McCartney a member of?,"[The Monkees, The Rolling Stones, The Yardbird...","[0.02, 0.02, 0.0, 0.96]",D,The Beatles,0.98,0.96,8611.0,4329.0
1,What is the short pleated skirt worn by Scotti...,"[Kelt, Shendyt, Glengarry, Kilt]","[0.204, 0.02, 0.0, 0.78]",D,Kilt,0.96,0.78,7763.0,4757.0
2,"What is the hard, white material sourced from ...","[Enamel, Ivory, Porcelain, Baleen]","[0.0, 1.0, 0.0, 0.0]",B,Ivory,0.94,1.00,6901.0,5265.0
3,What word means to trade by exchanging goods f...,"[Haggle, Swap, Export, Barter]","[0.02, 0.061, 0.0, 0.92]",D,Barter,0.93,0.92,8453.0,7562.0
4,What is an airplane without an engine called?,"[Biplane, Airship, Glider, Prop]","[0.041, 0.02, 0.92, 0.02]",C,Glider,0.89,0.92,6880.0,5656.0
...,...,...,...,...,...,...,...,...,...
416,Who invented the Christmas Cracker?,"[Tom Smith, Charles Plimpton, Frank Hornby, Ol...","[0.04, 0.367, 0.265, 0.327]",A,Tom Smith,0.00,0.04,NaN,3034.0
417,What was raced in the first competitive event ...,"[Balloons, Cars, Bicycles, Horses]","[0.02, 0.714, 0.163, 0.102]",A,Balloons,0.00,0.02,NaN,201.0
418,What do you call the accumulation of earth and...,"[Moraine, Cirque, Tarn, Esker]","[0.35, 0.102, 0.367, 0.184]",A,Moraine,0.00,0.35,NaN,11239.0
419,What is the name of the baseball player with t...,"[DiMaggio, Aaron, Cobb, Mays]","[0.224, 0.327, 0.27, 0.184]",C,Cobb,0.00,0.27,NaN,12323.0
